# 中国版全天候增强策略 - 回测分析

本Notebook用于复现华泰金工研报《从资产配置走向因子配置：中国版全天候增强策略》中的策略。

## 策略概述

1. **传统资产风险平价**：所有资产直接进行风险平价
2. **全天候基准策略**：将资产划分到四个宏观象限，先对四象限进行风险平价，象限内等权
3. **全天候增强策略**：引入预期共振动量，在增长和通胀维度各选择一个象限进行配置


## 1. 环境准备

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('.'))))

from source import DataLoader, RiskCalculator, StrategyBuilder, BacktestEngine, Visualizer
import config

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 2. 数据加载

In [ ]:
print("正在加载数据...")

data_loader = DataLoader(
    start_date=config.DATA_CONFIG['start_date'],
    end_date=config.DATA_CONFIG['end_date']
)

if config.DATA_CONFIG['use_saved_data'] and os.path.exists(f"{config.OUTPUT_CONFIG['output_dir']}/price_data.csv"):
    print("从本地加载数据...")
    data_loader = DataLoader.load_saved_data(config.OUTPUT_CONFIG['output_dir'])
else:
    print("从网络获取数据...")
    price_data, return_data = data_loader.load_all_data()
    data_loader.save_data(config.OUTPUT_CONFIG['output_dir'])

print(f"数据加载完成，共 {len(data_loader.return_data)} 个交易日")
print(f"可用资产: {list(data_loader.return_data.columns)}")

In [ ]:
print("价格数据预览:")
data_loader.price_data.head()

In [ ]:
print("收益率数据预览:")
data_loader.return_data.head()

## 3. 四象限组合分析

In [ ]:
strategy_builder = StrategyBuilder(data_loader)
quadrant_returns = strategy_builder.build_quadrant_portfolios(data_loader.return_data)
quadrant_performance = RiskCalculator.calculate_quadrant_performance(quadrant_returns)

quadrant_names = {
    'growth_above': '增长超预期',
    'growth_below': '增长不及预期',
    'inflation_above': '通胀超预期',
    'inflation_below': '通胀不及预期'
}

print("四象限组合绩效:")
quadrant_performance.rename(index=quadrant_names).style.format({
    '累计收益': '{:.2%}',
    '年化收益': '{:.2%}',
    '年化波动': '{:.2%}',
    '夏普比率': '{:.2f}',
    '最大回撤': '{:.2%}',
    '卡玛比率': '{:.2f}',
    '月度胜率': '{:.2%}'
})

In [ ]:
Visualizer.plot_quadrant_performance(quadrant_performance, title='四象限组合绩效对比')

In [ ]:
print("四象限净值曲线:")
(1 + quadrant_returns).cumprod().plot(figsize=(14, 7), title='四象限组合净值曲线',
                                      color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.ylabel('净值')
plt.grid(True, alpha=0.3)
plt.legend([quadrant_names.get(col, col) for col in quadrant_returns.columns])
plt.show()

## 4. 策略回测

In [ ]:
print("正在运行策略回测...")

backtest_engine = BacktestEngine(
    data_loader.return_data,
    rebalance_freq=config.BACKTEST_CONFIG['rebalance_freq'],
    fee_rate=config.BACKTEST_CONFIG['fee_rate']
)

results = backtest_engine.backtest_all_strategies(
    initial_capital=config.BACKTEST_CONFIG['initial_capital'],
    lookback_window=config.STRATEGY_CONFIG['lookback_window'],
    use_semicovariance=config.STRATEGY_CONFIG['use_semicovariance'],
    momentum_lookback=config.STRATEGY_CONFIG['momentum_lookback']
)

print("回测完成!")

## 5. 结果分析

In [ ]:
comparison = BacktestEngine.compare_strategies(results)

print("策略绩效对比:")
Visualizer.display_performance_table(comparison)

In [ ]:
Visualizer.plot_portfolio_value(results, title='策略净值对比')

In [ ]:
Visualizer.plot_drawdown(results, title='策略回撤对比')

In [ ]:
Visualizer.plot_performance_comparison(comparison, title='策略绩效对比')

## 6. 全天候基准策略分析

In [ ]:
Visualizer.plot_weights_evolution(
    results['allweather']['weights_record'],
    title='全天候基准策略 - 仓位演变'
)

In [ ]:
Visualizer.plot_asset_category_weights(
    results['allweather']['weights_record'],
    title='全天候基准策略 - 大类资产配置'
)

In [ ]:
Visualizer.plot_yearly_performance(
    results['allweather']['portfolio_returns'],
    title='全天候基准策略 - 年度收益'
)

## 7. 全天候增强策略分析

In [ ]:
Visualizer.plot_weights_evolution(
    results['enhanced']['weights_record'],
    title='全天候增强策略 - 仓位演变'
)

In [ ]:
Visualizer.plot_asset_category_weights(
    results['enhanced']['weights_record'],
    title='全天候增强策略 - 大类资产配置'
)

In [ ]:
Visualizer.plot_yearly_performance(
    results['enhanced']['portfolio_returns'],
    title='全天候增强策略 - 年度收益'
)

## 8. 结果保存

In [ ]:
import pickle

output_dir = config.OUTPUT_CONFIG['output_dir']
os.makedirs(output_dir, exist_ok=True)

with open(f'{output_dir}/backtest_results.pkl', 'wb') as f:
    pickle.dump(results, f)

comparison.to_csv(f'{output_dir}/performance_comparison.csv')

print(f"结果已保存到 {output_dir} 目录")

## 9. 总结

本项目完整复现了华泰金工研报《从资产配置走向因子配置：中国版全天候增强策略》中的策略框架，包括：

1. **数据获取**：使用akshare获取真实市场ETF数据
2. **四象限划分**：根据宏观逻辑将资产划分到四个象限
3. **风险平价**：使用EWMA半协方差矩阵计算风险，进行风险平价配置
4. **增强策略**：引入预期共振动量进行象限选择
5. **回测分析**：完整的回测引擎和可视化分析

通过对比三种策略，可以看到全天候增强策略在收益方面相比传统资产风险平价策略有明显提升，同时保持了较好的风险控制。